# Methode C — Vision-LLM-Transformation

**Concept:** The LLM receives a **screenshot** of the Figma mockup as its primary input and uses it to directly generate Vue 3 single-file components (SFCs) — without accessing the structured Figma metadata.

**Models:**

| Display name                    | OpenRouter model ID             | Provider   |
|---------------------------------|---------------------------------|------------|
| `anthropic/claude-sonnet-5`     | `anthropic/claude-sonnet-5`     | Anthropic  |
| `openai/gpt-5.6-terra`          | `openai/gpt-5.6-terra`          | OpenAI     |
| `google/gemini-3.1-pro-preview` | `google/gemini-3.1-pro-preview` | Google     |

=> `moonshotai/kimi-k2.6` not used anymore, because the requests are proceeded too slow (queued for minutes).

**Three contextual strategies (identical to Approach B):**

| Variant | Context                                     | Character                                       |
|---------|---------------------------------------------|-------------------------------------------------|
| **C1**  | No documentation                            | minimal, no api reference, error-prone          |
| **C2**  | Docs only for detected components (raw)     | focused, relevant, but noisy                    |
| **C3**  | Docs only for detected components (cleaned) | focused, relevant, clean, best expected results |

> **Methodological note:** Component recognition for C2/C3 continues to use the Figma JSON files as a preprocessing step to select relevant documents. The LLM prompt contains **only the screenshot**—no JSON. This is methodologically sound because the use of JSON is limited to context selection and does not serve as input for the transformation.

**Direct comparison with Approach B:** Identical model (`gpt-5.3-codex`), identical strategies, identical dataset — the only independent variable is the input format (JSON vs. screenshot).

In [33]:
import os
import re
import csv
import json
import time
import base64
import datetime
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

In [34]:
# 'components' or 'uis'
TYPE = 'components'

# 'messy' or 'pretty' (for ui test dataset)
VARIANT = ''

SCREENSHOTS_DIR = f'dataset/images/{TYPE}/' + (f'{VARIANT}/' if VARIANT else '')
FEW_SHOT_SCREENSHOTS_DIR = 'dataset/images/few-shot-examples'
FIGMA_JSON_DIR = f'dataset/figma-data/cleaned/{TYPE}/' + (f'{VARIANT}/' if VARIANT else '')

OUTPUT_DIR = f'dataset/storybook/src/code/{TYPE}/' + (f'{VARIANT}/' if VARIANT else '')

DOCS_DIR_RAW = 'primevue/component-documentation/raw'
DOCS_DIR_CLEANED = 'primevue/component-documentation/cleaned'

RUN_ID = 1

# 'zero_shot' or 'few_shot'
PROMPT_STRATEGY = 'few_shot'

API_URL = 'https://openrouter.ai/api/v1/chat/completions'

API_MODELS = {
    'claude-sonnet-5': 'anthropic/claude-sonnet-5',
    'gpt-5.6-terra': 'openai/gpt-5.6-terra',
    'gemini-3.1-pro-preview': 'google/gemini-3.1-pro-preview',
}

API_MAX_TOKENS = 8192
API_TEMPERATURE = 0.0
API_REASONING_EFFORT = 'medium'
IMAGE_DETAIL = 'auto'

load_dotenv(dotenv_path=Path('.env'))

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

if OPENROUTER_API_KEY is None:
    raise ValueError('OPENROUTER_API_KEY not found in environment variables. Please set it in the .env file.')

## Load data and documentations

### 1.1 Load Raw Component-Documentations

In [35]:
DOCS_DIR_PATH = Path(DOCS_DIR_RAW)
RAW_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    RAW_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(RAW_DOCS)}')
for name, content in RAW_DOCS.items():
    tokens_est = len(content) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_C2 = sum(len(c) // 4 for c in RAW_DOCS.values())
print(f'\nTotal C2-Context: ~{TOTAL_TOKENS_C2:,} Tokens')

Loaded Docs: 25
  accordion             ~ 8091 Tokens
  avatar                ~ 2737 Tokens
  badge                 ~ 1860 Tokens
  breadcrumb            ~  988 Tokens
  button                ~14672 Tokens
  card                  ~ 1426 Tokens
  checkbox              ~ 4414 Tokens
  datatable             ~20664 Tokens
  datepicker            ~11753 Tokens
  dialog                ~ 9667 Tokens
  divider               ~ 4877 Tokens
  inputnumber           ~ 7011 Tokens
  inputtext             ~ 6664 Tokens
  menu                  ~ 3240 Tokens
  password              ~ 8082 Tokens
  popover               ~ 3859 Tokens
  progressbar           ~ 1173 Tokens
  radiobutton           ~ 4283 Tokens
  select                ~11278 Tokens
  skeleton              ~ 3134 Tokens
  slider                ~ 2688 Tokens
  tabs                  ~ 7255 Tokens
  tag                   ~ 1966 Tokens
  textarea              ~ 6287 Tokens
  toggleswitch          ~ 2693 Tokens

Total C2-Context: ~150,762 Tokens

### 1.2 Load Cleand Component-Documentations

In [36]:
DOCS_DIR_PATH = Path(DOCS_DIR_CLEANED)
CLEANED_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    CLEANED_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(CLEANED_DOCS)}')
for name, content in CLEANED_DOCS.items():
    tokens_est = len(content) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_C3 = sum(len(c) // 4 for c in CLEANED_DOCS.values())
print(f'\nTotal C3-Context: ~{TOTAL_TOKENS_C3:,} Tokens')

missing_cleaned = set(RAW_DOCS) - set(CLEANED_DOCS)
missing_raw = set(CLEANED_DOCS) - set(RAW_DOCS)
assert not missing_cleaned, f'Missing cleaned docs for: {sorted(missing_cleaned)}'
assert not missing_raw, f'Missing raw docs for: {sorted(missing_raw)}'

Loaded Docs: 25
  accordion             ~ 1811 Tokens
  avatar                ~  432 Tokens
  badge                 ~  283 Tokens
  breadcrumb            ~  169 Tokens
  button                ~ 3820 Tokens
  card                  ~  501 Tokens
  checkbox              ~  626 Tokens
  datatable             ~ 2480 Tokens
  datepicker            ~ 1755 Tokens
  dialog                ~ 1349 Tokens
  divider               ~  725 Tokens
  inputnumber           ~ 1634 Tokens
  inputtext             ~ 3649 Tokens
  menu                  ~  539 Tokens
  password              ~ 4501 Tokens
  popover               ~  761 Tokens
  progressbar           ~  272 Tokens
  radiobutton           ~  571 Tokens
  select                ~ 2065 Tokens
  skeleton              ~  438 Tokens
  slider                ~  487 Tokens
  tabs                  ~ 1641 Tokens
  tag                   ~  360 Tokens
  textarea              ~ 3549 Tokens
  toggleswitch          ~  587 Tokens

Total C3-Context: ~35,005 Tokens


### 1.3 Load Figma JSON Data

In [37]:
INPUT_JSON_DATA_DIR_PATH = Path(FIGMA_JSON_DIR)
FIGMA_DATA: dict[str, dict] = {}

json_data_input_files = sorted(
    f for f in INPUT_JSON_DATA_DIR_PATH.rglob('*.json')
    if f.parent != INPUT_JSON_DATA_DIR_PATH
)

for input_file in json_data_input_files:
    print(f'Loading {input_file}...')

    with open(input_file, encoding='utf-8-sig') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()

        FIGMA_DATA[key] = json.load(f)

print(f'\nLoaded Figma JSONs: {len(FIGMA_DATA)}')
for name, data in FIGMA_DATA.items():
    tokens_est = len(json.dumps(data)) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_FIGMA = sum(len(json.dumps(d)) // 4 for d in FIGMA_DATA.values())
print(f'\nTotal Figma JSON Tokens: ~{TOTAL_TOKENS_FIGMA:,} Tokens')

Loading dataset\figma-data\cleaned\components\hard\1.json...
Loading dataset\figma-data\cleaned\components\hard\10.json...
Loading dataset\figma-data\cleaned\components\hard\2.json...
Loading dataset\figma-data\cleaned\components\hard\3.json...
Loading dataset\figma-data\cleaned\components\hard\4.json...
Loading dataset\figma-data\cleaned\components\hard\5.json...
Loading dataset\figma-data\cleaned\components\hard\6.json...
Loading dataset\figma-data\cleaned\components\hard\7.json...
Loading dataset\figma-data\cleaned\components\hard\8.json...
Loading dataset\figma-data\cleaned\components\hard\9.json...
Loading dataset\figma-data\cleaned\components\medium\1.json...
Loading dataset\figma-data\cleaned\components\medium\10.json...
Loading dataset\figma-data\cleaned\components\medium\2.json...
Loading dataset\figma-data\cleaned\components\medium\3.json...
Loading dataset\figma-data\cleaned\components\medium\4.json...
Loading dataset\figma-data\cleaned\components\medium\5.json...
Loading da

### 1.4 Load Screenshots

In [38]:
SCREENSHOTS_DIR_PATH = Path(SCREENSHOTS_DIR)
SCREENSHOTS: dict[str, str] = {}

all_screenshot_input_files = sorted(f for f in SCREENSHOTS_DIR_PATH.rglob('*.png')
                         if f.parent != SCREENSHOTS_DIR_PATH)

for input_file in all_screenshot_input_files:
    print(f'Loading {input_file}...')

    with open(input_file, 'rb') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()

        SCREENSHOTS[key] = base64.b64encode(f.read()).decode('utf-8')

print(f'\nLoaded Screenshots: {len(SCREENSHOTS)}')
for name, content in SCREENSHOTS.items():
    tokens_est = len(content) // 4
    size_kb = len(content) / 1024

    print(f'  {name:20s} ~{tokens_est:5d} Tokens  ({size_kb:.1f} KB)')

TOTAL_TOKENS_SCREENSHOTS = sum(len(c) // 4 for c in SCREENSHOTS.values())
print(f'\nTotal Figma Screenshot Tokens: ~{TOTAL_TOKENS_SCREENSHOTS:,} Tokens')

Loading dataset\images\components\hard\1.png...
Loading dataset\images\components\hard\10.png...
Loading dataset\images\components\hard\2.png...
Loading dataset\images\components\hard\3.png...
Loading dataset\images\components\hard\4.png...
Loading dataset\images\components\hard\5.png...
Loading dataset\images\components\hard\6.png...
Loading dataset\images\components\hard\7.png...
Loading dataset\images\components\hard\8.png...
Loading dataset\images\components\hard\9.png...
Loading dataset\images\components\medium\1.png...
Loading dataset\images\components\medium\10.png...
Loading dataset\images\components\medium\2.png...
Loading dataset\images\components\medium\3.png...
Loading dataset\images\components\medium\4.png...
Loading dataset\images\components\medium\5.png...
Loading dataset\images\components\medium\6.png...
Loading dataset\images\components\medium\7.png...
Loading dataset\images\components\medium\8.png...
Loading dataset\images\components\medium\9.png...
Loading dataset\im

### 1.5 Load Few-Shot-Example-Screenshots

In [39]:
FEW_SHOT_SCREENSHOTS_DIR_PATH = Path(FEW_SHOT_SCREENSHOTS_DIR)
FEW_SHOT_SCREENSHOTS: dict[str, str] = {}

for png_file in sorted(FEW_SHOT_SCREENSHOTS_DIR_PATH.glob('few-shot-example-*-composite.png')):
    key = png_file.stem.removeprefix('few-shot-example-').removesuffix('-composite')  # 'simple' | 'medium' | 'hard'

    with open(png_file, 'rb') as f:
        FEW_SHOT_SCREENSHOTS[key] = base64.b64encode(f.read()).decode('utf-8')

print(f'Loaded few-shot screenshots: {list(FEW_SHOT_SCREENSHOTS.keys())}')

if PROMPT_STRATEGY == 'few_shot' and len(FEW_SHOT_SCREENSHOTS) < 3:
    missing = {'simple', 'medium', 'hard'} - set(FEW_SHOT_SCREENSHOTS.keys())

    raise ValueError(
        f'few_shot-Modus active, but Screenshots missing for: {sorted(missing)}. '
        f'Expected under {FEW_SHOT_SCREENSHOTS_DIR}/few-shot-example-<level>-composite.png'
    )

missing_screenshots = set(FIGMA_DATA) - set(SCREENSHOTS)
missing_json = set(SCREENSHOTS) - set(FIGMA_DATA)
assert not missing_screenshots, f'Screenshots fehlen für: {sorted(missing_screenshots)}'
assert not missing_json, f'Figma-JSON fehlt für: {sorted(missing_json)}'

Loaded few-shot screenshots: ['hard', 'medium', 'simple']


## 2. Detect PrimeVue-Components in Figma Data

In [40]:
KNOWN_COMPONENTS = set(CLEANED_DOCS.keys())

# Aliases (Figma-Name -> Doc-Name)
DOC_ALIASES = {
    'calendar': 'datepicker',
    'overlaybadge': 'badge',
}


def _normalize(name: str) -> str:
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def detect_components(figma_node: dict, found: set | None = None) -> set[str]:
    """Collect all PrimeVue component names that appear in the Figma JSON"""
    if found is None:
        found = set()

    if not isinstance(figma_node, dict):
        return found

    name = figma_node.get('name', '')
    t = figma_node.get('type', '')

    if name.startswith('_'):
        return found  # interne Sub-Instance

    norm = _normalize(name)
    norm = DOC_ALIASES.get(norm, norm)

    if t in ('INSTANCE', 'FRAME') and norm in KNOWN_COMPONENTS:
        found.add(norm)

    for child in figma_node.get('children', []) or []:
        detect_components(child, found)

    return found


DETECTED_COMPONENTS: dict[str, set[str]] = {}

for key, data in FIGMA_DATA.items():
    detected = detect_components(data)
    DETECTED_COMPONENTS[key] = detected

    print(f'{key:20s}  -> Detected: {", ".join(sorted(detected)) or "None"}')

print(f'\nTotal Unique Detected Components: {len(set.union(*DETECTED_COMPONENTS.values()))}')

hard-1                -> Detected: button, checkbox, dialog, inputtext, select
hard-10               -> Detected: button, datatable, popover
hard-2                -> Detected: datatable
hard-3                -> Detected: button, datepicker, inputnumber, inputtext, select
hard-4                -> Detected: button, datatable, popover
hard-5                -> Detected: button, dialog, tabs
hard-6                -> Detected: inputtext, select
hard-7                -> Detected: avatar, button, card, divider, tag
hard-8                -> Detected: button, card, tabs
hard-9                -> Detected: avatar, breadcrumb, button, datepicker, dialog, divider, inputtext, textarea
medium-1              -> Detected: button, card, checkbox, inputtext, password
medium-10             -> Detected: button, card, progressbar
medium-2              -> Detected: tabs
medium-3              -> Detected: accordion
medium-4              -> Detected: breadcrumb, button, menu
medium-5              -> Detected: b

## 3. Prompt Building

### 3.1 Context-Builder

In [41]:
def build_context(strategy: str, figma_node: dict | None = None) -> tuple[str, list[str]]:
    """Returns (context_string, used_components)

    strategy: 'c1' | 'c2' | 'c3'
    figma_node: Required c2 and c3 for component recognition
    """

    if strategy == 'c1':
        return '', []  # No context for C1

    if figma_node is None:
        raise ValueError('Strategies requires figma_node for component recognition.')

    docs = RAW_DOCS if strategy == 'c2' else CLEANED_DOCS

    detected_components = detect_components(figma_node)

    context_parts = []
    used_components = []

    for comp in sorted(detected_components):
        if comp in docs:
            print(f'  Adding doc for component: {comp} ({len(docs[comp])//4} tokens)')

            context_parts.append(f'# {comp}\n\n{docs[comp]}')
            used_components.append(comp)

    return '\n\n'.join(context_parts), used_components # Only docs of detected components in raw or cleaned form, depending on strategy

### 3.2 Prompt Templates

In [42]:
SYSTEM_INSTRUCTIONS = """You are an expert Vue 3 and PrimeVue developer.
Analyze the given Figma mockup UI screenshot and transform it into a complete, working Vue 3 Single File Component with PrimeVue 4 components. Use the provided PrimeVue documentation if given as reference for component usage and props.

STRICT REQUIREMENTS:
- Use PrimeVue 4 components exclusively for all UI elements visible in the screenshot
- Use <script setup> syntax (no Options API)
- Import every PrimeVue component used: import Button from 'primevue/button'
- Use Tailwind CSS utility classes for layout and spacing
- Use ref() from Vue for all form/input state
 Infer layout direction (row/column), spacing, and alignment visually from the image; map to flex/flex-col, gap-*, p-*/px-*/py-* using the closest sensible Tailwind scale value
- Infer component variants (e.g. severity, size, outlined/filled) from visual styling (color, border, fill) — do not invent props that aren't visually supported
- Output ONLY the Vue SFC — no explanation, no markdown fences, no prose
- Return exactly one complete Vue SFC, starting directly with <template>
  and ending with </script>
- If a region of the image is ambiguous, low-resolution, or partially occluded, use the closest valid structural mapping supported by what IS visible; do not invent unsupported behavior
- Treat the transformation as incomplete until all eligible non-ignored nodes are represented in the output
- Before finalizing, verify that the SFC is syntactically valid, all used
  PrimeVue components are imported, and all form/input state uses ref()
- Assume PrimeVue Aura theme as baseline for styling; do not generate custom theme CSS unless explicitly required by Figma mockup UI screenshot

WHAT TO ANALYZE IN THE SCREENSHOT:
- Identify all UI components (inputs, buttons, tables, cards, dialogs, etc.)
- Detect the layout structure (rows, columns, nested containers, spacing)
- Extract all text content (labels, placeholders, button labels, headings)
- Identify component states (disabled, selected, checked, invalid, etc.)
- Recognize component variants (outlined, filled, severity colors, sizes)
- Map the visual hierarchy to a PrimeVue component structure

IMAGE INTERPRETATION GUIDE:
- Repeated visual patterns (cards, list rows, form fields) usually correspond to one reusable PrimeVue component instance each
- Icons should be mapped to PrimeIcons (pi pi-*) by closest visual match
- Text content shown in the image must be transcribed exactly as rendered, not paraphrased or translated"""

FEW_SHOT_VUE_EXAMPLES: dict[str, str] = {
    'simple': '''<template>
  <div class="flex w-lg flex-col gap-4 p-6">
    <div class="flex items-center gap-4">
      <Avatar label="B" size="xlarge" shape="circle" />
      <span class="text-xl text-black">Benutzername</span>
    </div>
    <Textarea v-model="feedback" placeholder="Feedback eingeben..." />
    <div class="flex items-center justify-between">
      <div class="flex items-center gap-2">
        <Checkbox v-model="notification" input-id="notification" binary />
        <label for="notification">Benachrichtigen</label>
      </div>
      <Button label="Senden" severity="primary" class="w-fit" />
    </div>
  </div>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Avatar from 'primevue/avatar'
  import Button from 'primevue/button'
  import Textarea from 'primevue/textarea'
  import Checkbox from 'primevue/checkbox'

  const feedback = ref('')
  const notification = ref(false)
</script>''',
    'medium': '''<template>
  <Card
    :pt="{
      root: 'w-md p-8 gap-6',
      body: 'flex flex-col gap-4 !p-0',
      content: 'flex flex-col gap-4',
      footer: 'mt-2',
    }"
  >
    <template #header>
      <h1 class="text-lg font-medium">Anmelden</h1>
    </template>
    <template #content>
      <InputText v-model="email" type="email" placeholder="E-Mail-Adresse" input-id="email-input" />
      <Password v-model="password" input-id="password-input" toggle-mask input-class="w-full" />
      <div class="flex items-center justify-between">
        <Badge value="Beliebt" severity="info" />
        <ProgressBar :value="50" :show-value="false" class="!h-1 w-[84px]" />
      </div>
    </template>
    <template #footer>
      <Button label="Jetzt starten" severity="primary" class="w-full" />
    </template>
  </Card>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Card from 'primevue/card'
  import Badge from 'primevue/badge'
  import Button from 'primevue/button'
  import Password from 'primevue/password'
  import InputText from 'primevue/inputtext'
  import ProgressBar from 'primevue/progressbar'

  const email = ref('')
  const password = ref('password')
</script>''',
    'hard': '''<template>
  <DataTable :value="projects">
    <Column field="name" header="Name" />
    <Column field="status" header="Status">
      <template #body="{ data }">
        <Tag :value="data.status" :severity="getStatusTagSeverity(data.status)" />
      </template>
    </Column>
    <Column header="Aktionen" header-class="w-24" body-class="w-24 flex justify-center">
      <template #body>
        <Button
          icon="pi pi-ellipsis-h"
          severity="secondary"
          aria-haspopup="true"
          aria-controls="actions-menu"
          @click="actionsMenu?.toggle"
        />
      </template>
    </Column>
  </DataTable>
  <Menu
    ref="actions-menu"
    id="actions-menu"
    :model="actionOptions"
    popup
    :pt="{
      list: 'flex flex-col !gap-2 !p-2.5',
    }"
  >
    <template #item="{ item }">
      <Button
        :label="item.label"
        :icon="item.icon"
        severity="secondary"
        outlined
        class="w-full !justify-start"
      />
    </template>
  </Menu>
  <Dialog
    v-model:visible="isEditProjektDialogVisible"
    header="Projekt bearbeiten"
    modal
    :pt="{
      root: 'w-full max-w-md',
      content: 'flex flex-col !gap-4',
    }"
  >
    <div class="flex flex-col gap-2">
      <label for="name-input" class="text-sm">Name</label>
      <InputText v-model="name" type="text" input-id="name-input" />
    </div>
    <template #footer>
      <Button label="Abbrechen" severity="secondary" />
      <Button label="Speichern" severity="primary" />
    </template>
  </Dialog>
</template>

<script setup lang="ts">
  import { ref, useTemplateRef } from 'vue'
  import Tag from 'primevue/tag'
  import Column from 'primevue/column'
  import DataTable from 'primevue/datatable'
  import Button from 'primevue/button'
  import Menu from 'primevue/menu'
  import Dialog from 'primevue/dialog'
  import InputText from 'primevue/inputtext'

  const projects = ref([
    {
      name: 'Webseite Relaunch',
      status: 'Aktiv',
    },
  ])

  const isEditProjektDialogVisible = ref(true)
  const name = ref('Webseite Relaunch')

  const actionsMenu = useTemplateRef('actions-menu')
  const actionOptions = [
    {
      label: 'Bearbeiten',
      icon: 'pi pi-pen-to-square',
      command: () => (isEditProjektDialogVisible.value = true),
    },
    {
      label: 'Löschen',
      icon: 'pi pi-trash',
    },
  ]

  function getStatusTagSeverity(status: string) {
    switch (status) {
      case 'Aktiv':
        return 'success'
      case 'In Prüfung':
        return 'warn'
      case 'Abgeschlossen':
        return 'info'
      case 'Gestoppt':
        return 'danger'
    }
  }
</script>''',
}

DOCS_MESSAGE_PROMPT = """PrimeVue documentation for reference:
{context}"""

USER_PROMPT = """Transform the following Figma mockup UI screenshot into a Vue 3 Single File Component using PrimeVue components."""

### 3.3 Prompt Builder

In [43]:
def _image_block(base64_png: str) -> dict:
    """OpenAI-compatible image content block (works via OpenRouter for all models)."""
    return {
        'type': 'image_url',
        'image_url': {
            'url': f'data:image/png;base64,{base64_png}',
            'detail': IMAGE_DETAIL,
        },
    }


def build_few_shot_turns() -> list[dict]:
    """Builds the few-shot example turns (user: screenshot, assistant: SFC).

    The LAST content block of the LAST assistant turn carries a cache_control
    marker (Breakpoint 2): prefix-based caching then covers instructions +
    all few-shot turns including the screenshot image tokens.
    """
    turns: list[dict] = []

    for level in ('simple', 'medium', 'hard'):
        screenshot = FEW_SHOT_SCREENSHOTS.get(level)
        vue_code = FEW_SHOT_VUE_EXAMPLES.get(level)

        if screenshot is None or vue_code is None:
            print(f'  WARNING: Few-Shot-Examples "{level}" missing, skipping.')

            continue

        turns.append({
            'role': 'user',
            'content': [
                _image_block(screenshot),
                {'type': 'text', 'text': USER_PROMPT},
            ],
        })

        turns.append({
            'role': 'assistant',
            'content': [
                {'type': 'text', 'text': f'```vue\n{vue_code}\n```'},
            ],
        })

    if turns:
        turns[-1]['content'][-1]['cache_control'] = {'type': 'ephemeral'}  # Breakpoint 2

    return turns


FEW_SHOT_TURNS = build_few_shot_turns() if PROMPT_STRATEGY == 'few_shot' else []
print(f'Few-shot turns built: {len(FEW_SHOT_TURNS)} ({len(FEW_SHOT_TURNS)//2} examples)')

# TODO - consider that context of prime vue components influences output ("einzige unabhängige Variable ist die Eingabemodalität")
def build_messages(figma_root: dict, screenshot: str, strategy: str) -> tuple[list[dict], list[str], int]:
    """Creates the OpenRouter message list with cache breakpoints.

    Structure (static -> variable, prefix-based caching):
      1. System message:  instructions only       -> Breakpoint 1 (static across ALL calls)
      2. Few-shot turns:  user/assistant pairs    -> Breakpoint 2 on last block (FEW-SHOT only)
      3. User block 1:    PrimeVue docs (c2/c3)   -> Breakpoint 3 (repeats per doc combination)
      4. User blocks 2+3: mockup screenshot + task -> variable per call, never cached

    Returns: (messages, used_components, context_tokens)
    """
    context, used_components = build_context(strategy, figma_root)
    context_tokens = len(context) // 4

    user_content = []

    if context:
        user_content.append({
            'type': 'text',
            'text': DOCS_MESSAGE_PROMPT.format(context=context),
            'cache_control': {'type': 'ephemeral'},      # Breakpoint 3 (c2/c3)
        })

    user_content.append(_image_block(screenshot))
    user_content.append({'type': 'text', 'text': USER_PROMPT})

    messages = [
        {'role': 'system', 'content': [{
            'type': 'text',
            'text': SYSTEM_INSTRUCTIONS,
            'cache_control': {'type': 'ephemeral'},      # Breakpoint 1
        }]},
        *FEW_SHOT_TURNS,
        {'role': 'user', 'content': user_content},
    ]

    return messages, used_components, context_tokens

Few-shot turns built: 6 (3 examples)


## 4. LLM Interaction

In [44]:
def call_llm_vision(messages: list[dict], strategy: str, key: str, model_id: str) -> dict:
    """Calls the OpenRouter Chat Completions API (OpenAI-compatible format,
    works uniformly across OpenAI-, Anthropic- and Google-models).

    model_id: OpenRouter model id, e.g. 'anthropic/claude-sonnet-5'

    Returns: {
        'content': str,
        'generation_id': str,
        'provider': str,
        'model': str,
        'input_tokens': int,
        'output_tokens': int,
        'total_tokens': int,
        'cached_tokens': int,
        'cache_write_tokens': int,
        'reasoning_tokens': int,
        'cost_usd': float,
        'cost_prompt_usd': float,
        'cost_completion_usd': float,
        'stop_reason': str,
        'native_finish_reason': str,
        'refusal': str | None,
        'duration': float,
    }
    """
    metadata = {
        'strategy':   strategy,
        'mockup_key': key,
        'model':      model_id,
    }

    payload = json.dumps({
        'model': model_id,
        'temperature': API_TEMPERATURE,
        'max_tokens': API_MAX_TOKENS,
        'reasoning': {'effort': API_REASONING_EFFORT},
        'metadata': metadata,
        'usage': {'include': True},
        'session_id': f'{model_id}/{strategy}/{PROMPT_STRATEGY}',
        'messages': messages,
    }).encode('utf-8')

    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
    }

    req = urllib.request.Request(
        API_URL,
        data=payload,
        headers=headers,
        method='POST',
    )

    start_time = time.time()

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        error_text = e.read().decode('utf-8', errors='ignore')
        try:
            detail = json.loads(error_text).get('error', {}).get('message', error_text)
        except json.JSONDecodeError:
            detail = error_text or str(e)
        raise RuntimeError(f'OpenRouter API Fehler {e.code}: {detail}') from e
    except urllib.error.URLError as e:
        raise RuntimeError(f'Netzwerkfehler: {e.reason}') from e

    end_time = time.time()

    choice = resp_data['choices'][0]
    usage = resp_data.get('usage', {})
    prompt_details = usage.get('prompt_tokens_details', {})
    completion_details = usage.get('completion_tokens_details', {})
    cost_details = usage.get('cost_details', {})

    content = choice['message'].get('content')

    if content is None:
        raise RuntimeError(f'OpenRouter API returned no content: {resp_data}')

    return {
        'provider':             resp_data.get('provider'),
        'model':                resp_data.get('model'),

        'generation_id':        resp_data.get('id'),

        'content':              content,

        'input_tokens':         usage.get('prompt_tokens', 0),
        'output_tokens':        usage.get('completion_tokens', 0),
        'total_tokens':         usage.get('total_tokens', 0),

        'cached_tokens':        prompt_details.get('cached_tokens', 0),
        'cache_write_tokens':   prompt_details.get('cache_write_tokens', 0),
        'reasoning_tokens':     completion_details.get('reasoning_tokens', 0),

        'cost_usd':             usage.get('cost'),
        'cost_prompt_usd':      cost_details.get('upstream_inference_prompt_cost'),
        'cost_completion_usd':  cost_details.get('upstream_inference_completions_cost'),

        'stop_reason':          choice.get('finish_reason', ''),
        'native_finish_reason': choice.get('native_finish_reason', ''),
        'refusal':              choice['message'].get('refusal'),

        'duration':             end_time - start_time,
    }

In [45]:
def extract_sfc(raw: str) -> tuple[str, bool]:
    """Returns (sfc, extraction_ok)."""
    blocks = re.findall(r'```(?:vue)?\s*\n(.+?)```', raw, re.DOTALL)
    candidates = [b.strip() for b in blocks if '<template>' in b]

    if candidates:
        longest = max(candidates, key=len)

        return _trim_to_last_close_tag(longest), True

    if '<template>' in raw:
        start = raw.index('<template>')

        return _trim_to_last_close_tag(raw[start:].strip()), True

    return f'<!-- SFC-Extraktion failed -->\n<!-- RAW:\n{raw[:500]}\n-->', False


def _trim_to_last_close_tag(sfc: str) -> str:
    for tag in ('</style>', '</script>'):
        idx = sfc.rfind(tag)

        if idx != -1:
            return sfc[: idx + len(tag)]

    return sfc

## 5. Record metrics

In [46]:
_metrics_c: dict = {}

def _reset_metrics_c():
    global _metrics_c

    _metrics_c = {
        'provider': '',
        'model': '',
        'model_reported': '',

        'generation_id': '',

        'context_tokens': 0,
        'context_components': 0,

        'duration': 0,

        'input_tokens': 0,
        'output_tokens': 0,
        'total_tokens': 0,
        'cached_tokens': 0,
        'cache_write_tokens': 0,
        'reasoning_tokens': 0,

        'cost_usd': 0,
        'cost_prompt_usd': 0,
        'cost_completion_usd': 0,

        'sfc_bytes': 0,
        'sfc_lines': 0,

        'extraction_ok': False,
        'parse_ok': False,
        'truncated': False,

        'stop_reason': '',
        'native_finish_reason': '',
        'refusal': 0,
    }

## 6. Main-Transformation for one Screenshot with a given strategy

In [47]:
def generate_sfc_c(figma_root: dict, screenshot: str, strategy: str, key: str, model_id: str) -> str:
    """Transforms a Figma mockup via screenshot using the specified LLM strategy and model.

    strategy: 'c1' | 'c2' | 'c3'
    Returns: Vue-3-SFC als String
    """
    _reset_metrics_c()

    messages, used_components, context_tokens = build_messages(figma_root, screenshot, strategy)

    print(f'Detected Components: {used_components} -> Context Tokens: {context_tokens}')

    response = call_llm_vision(messages, strategy, key, model_id)

    _metrics_c['provider']       = response['provider']
    _metrics_c['model']          = response['model']
    _metrics_c['model_reported'] = response['model']

    _metrics_c['generation_id'] = response['generation_id']

    _metrics_c['context_tokens']     = context_tokens
    _metrics_c['context_components'] = len(used_components)

    _metrics_c['duration'] = response['duration']

    _metrics_c['input_tokens']  = response['input_tokens']
    _metrics_c['output_tokens'] = response['output_tokens']
    _metrics_c['total_tokens'] = response['total_tokens']
    _metrics_c['cached_tokens']  = response['cached_tokens']
    _metrics_c['cache_write_tokens'] = response['cache_write_tokens']
    _metrics_c['reasoning_tokens'] = response['reasoning_tokens']

    _metrics_c['cost_usd'] = response['cost_usd']
    _metrics_c['cost_prompt_usd'] = response['cost_prompt_usd']
    _metrics_c['cost_completion_usd'] = response['cost_completion_usd']

    _metrics_c['stop_reason'] = response['stop_reason']
    _metrics_c['native_finish_reason'] = response['native_finish_reason']
    _metrics_c['refusal'] = response['refusal']

    sfc, extraction_ok = extract_sfc(response['content'])

    _metrics_c['sfc_bytes'] = len(sfc)
    _metrics_c['sfc_lines'] = sfc.count('\n') + 1

    _metrics_c['extraction_ok'] = extraction_ok
    _metrics_c['parse_ok'] = '<template>' in sfc and '<script' in sfc
    _metrics_c['truncated'] = _metrics_c['native_finish_reason'] in {'MAX_TOKENS', 'length'}

    return sfc

## 7. Pipeline for all Screenshots and all strategies

In [48]:
STRATEGIES = ['c1', 'c2', 'c3']
OUTPUT_PATH = Path(OUTPUT_DIR)

RESULTS_CSV_PATH = Path('results') / (
    f'results_c_{TYPE}_{VARIANT + "_" if VARIANT else ""}{PROMPT_STRATEGY}.csv'
)
RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

RESULT_FIELDNAMES = [
    'input',
    'output',

    'type',
    'variant',
    'prompt_strategy',

    'temperature',
    'reasoning_effort',
    'image_detail',

    'strategy',
    'complexity',

    'provider',
    'model',
    'model_reported',

    'run',
    'attempt',
    'generation_id',

    'context_tokens',
    'context_components',

    'duration',

    'input_tokens',
    'output_tokens',
    'total_tokens',
    'cached_tokens',
    'cache_write_tokens',
    'reasoning_tokens',

    'cost_usd',
    'cost_prompt_usd',
    'cost_completion_usd',

    'extraction_ok',
    'parse_ok',
    'truncated',

    'sfc_bytes',
    'sfc_lines',

    'stop_reason',
    'native_finish_reason',
    'refusal',
    'error',

    'created_at',
]

_MODEL_NAMES = list(API_MODELS.keys())


def _load_completed_keys(csv_path: Path) -> set[tuple[str, str, str, str, str]]:
    """Reads a previous results CSV (if any) and returns the set of
    (model, strategy, input, variant, run) combinations that already succeeded.
    Errored attempts are NOT counted as completed, so they get retried."""
    completed: set[tuple[str, str, str, str, str]] = set()

    if not csv_path.exists():
        return completed

    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row.get('error') in (None, '', 'None'):
                completed.add((row['model'], row['strategy'], row['input'], row['variant'], row['run']))

    return completed


def _append_result(csv_path: Path, result: dict) -> None:
    """Appends a single result row to the CSV, writing the header once."""
    is_new_file = not csv_path.exists()

    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDNAMES)

        if is_new_file:
            writer.writeheader()

        writer.writerow(result)


def _complexity_from_key(key: str) -> str:
    """Extract complexity prefix from keys like 'medium-8' or 'simple-5'."""
    return key.split('-', 1)[0] if '-' in key else 'unknown'


already_completed = _load_completed_keys(RESULTS_CSV_PATH)
if already_completed:
    print(f'Resuming: {len(already_completed)} combinations already completed in {RESULTS_CSV_PATH}\n')

print(f'Input-Files: {len(FIGMA_DATA)} Strategies: {STRATEGIES}\n')

processed_count = 0

for model_name, model_id in API_MODELS.items():

    print(f'\nUsing model: {model_name} ({model_id})\n{"="*60}')

    for strategy in STRATEGIES:

        print(f' Strategy: {strategy.upper()}')

        for key, figma_root in FIGMA_DATA.items():
            complexity = _complexity_from_key(key)

            if (model_name, strategy, key, VARIANT, str(RUN_ID)) in already_completed:
                print(f'  SKIP (already done) {key:25s} [{model_name}/{strategy}]')

                continue

            print(f' Processing {key}...\n{"="*60}')

            try:
                sfc = generate_sfc_c(figma_root, SCREENSHOTS[key], strategy, key, model_id)

                out_path = OUTPUT_PATH / 'c' / PROMPT_STRATEGY / complexity / \
                    f'{key.split("-", 1)[1]}-{strategy}-{model_name}-{RUN_ID}.vue'
                out_path.parent.mkdir(parents=True, exist_ok=True)
                out_path.write_text(sfc, encoding='utf-8')

                m = dict(_metrics_c)
                result = {
                    'input':                key,
                    'output':               out_path.name,

                    'type':                 TYPE,
                    'variant':              VARIANT,
                    'prompt_strategy':      PROMPT_STRATEGY,

                    'temperature':          API_TEMPERATURE,
                    'reasoning_effort':     API_REASONING_EFFORT,
                    'image_detail':         IMAGE_DETAIL,

                    'strategy':             strategy,
                    'complexity':           complexity,

                    'provider':             m['provider'],
                    'model':                model_name,
                    'model_reported':       m['model_reported'],

                    'run':                  RUN_ID,
                    'attempt':              1,
                    'generation_id':        m['generation_id'],

                    'context_tokens':       m['context_tokens'],
                    'context_components':   m['context_components'],

                    'duration':             round(m['duration'] * 1000, 4),

                    'input_tokens':         m['input_tokens'],
                    'output_tokens':        m['output_tokens'],
                    'total_tokens':         m['total_tokens'],
                    'cached_tokens':        m['cached_tokens'],
                    'cache_write_tokens':   m['cache_write_tokens'],
                    'reasoning_tokens':     m['reasoning_tokens'],

                    'cost_usd':             m['cost_usd'],
                    'cost_prompt_usd':      m['cost_prompt_usd'],
                    'cost_completion_usd':  m['cost_completion_usd'],

                    'extraction_ok':        m['extraction_ok'],
                    'parse_ok':             m['parse_ok'],
                    'truncated':            m['truncated'],

                    'sfc_bytes':            m['sfc_bytes'],
                    'sfc_lines':            m['sfc_lines'],

                    'stop_reason':          m['stop_reason'],
                    'native_finish_reason': m['native_finish_reason'],
                    'refusal':              m['refusal'],

                    'error':                None,

                    'created_at':           datetime.datetime.now().isoformat(),
                }

                print(f'  OK  {key:25s} [{strategy}]  '
                      f'in={m["input_tokens"]:5d}tok  '
                      f'cached={m["cached_tokens"]:5d}tok  '
                      f'out={m["output_tokens"]:4d}tok  '
                      f'${(result["cost_usd"] or 0):.4f}  '
                      f'{m["duration"]:6.0f}ms')

            except Exception as e:
                print(f'  ERROR {key:25s} [{model_name}/{strategy}]  {str(e)}')

                result = {k: None for k in RESULT_FIELDNAMES}

                result.update({
                    'input': key, 'output': None, 'type': TYPE, 'variant': VARIANT,
                    'prompt_strategy': PROMPT_STRATEGY, 'temperature': API_TEMPERATURE,
                    'reasoning_effort': API_REASONING_EFFORT, 'image_detail': IMAGE_DETAIL,
                    'strategy': strategy, 'complexity': complexity, 'run': RUN_ID, 'attempt': 1,
                    'error': str(e), 'created_at': datetime.datetime.now().isoformat(),
                })

            _append_result(RESULTS_CSV_PATH, result)

            processed_count += 1


print(f'\nProcessed {processed_count} new transformations this run.')

Input-Files: 30 Strategies: ['c1', 'c2', 'c3']


Using model: claude-sonnet-5 (anthropic/claude-sonnet-5)
 Strategy: C1
 Processing hard-1...
Detected Components: [] -> Context Tokens: 0
  OK  hard-1                    [c1]  in= 6667tok  cached=    0tok  out= 572tok  $0.0220       7ms
 Processing hard-10...
Detected Components: [] -> Context Tokens: 0
  OK  hard-10                   [c1]  in= 6378tok  cached= 5837tok  out=1385tok  $0.0161      12ms
 Processing hard-2...
Detected Components: [] -> Context Tokens: 0
  OK  hard-2                    [c1]  in= 6191tok  cached= 5837tok  out= 597tok  $0.0078       8ms
 Processing hard-3...
Detected Components: [] -> Context Tokens: 0
  OK  hard-3                    [c1]  in= 6331tok  cached= 5837tok  out= 680tok  $0.0090       8ms
 Processing hard-4...
Detected Components: [] -> Context Tokens: 0
  OK  hard-4                    [c1]  in= 6219tok  cached= 5837tok  out=1060tok  $0.0125      10ms
 Processing hard-5...
Detected Components: [] -> 